<a href="https://colab.research.google.com/github/VanDar1011/Copy_Folder_Google_Drive_to_Google_Drive/blob/main/Copy_Folder_Google_Drive_to_Google_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
from ipywidgets import widgets

dest_text = widgets.Text(description="Your drive", placeholder='Nhập đường link folder Google Drive của bạn')
source_text = widgets.Text(description="Shared drive", placeholder='Nhập đường link folder Google Drive shared')
from_page_text = widgets.Text(description="Từ trang", value="0")
to_page_text = widgets.Text(description="Đến trang", value="0")
max_download_size_text = widgets.Text(description="Tổng dung lượng tối đa(GB)", value="700")
exclude_str_text = widgets.Text(description="Bỏ file, folder có chứa nội dung", value="")

start_button = widgets.Button(description="Bắt đầu")
output = widgets.Output()

def on_start_button_clicked(b):
    global destDriveLink, sourceDriveLink, fromPage, toPage, maxDownloadSize, excludeStr
    destDriveLink = dest_text.value
    sourceDriveLink = source_text.value
    try:
        fromPage = int(from_page_text.value)
        toPage = int(to_page_text.value)
        maxDownloadSize = int(max_download_size_text.value)
    except ValueError:
        with output:
            print("Please enter valid numbers for 'Từ trang', 'Đến trang', and 'Tổng dung lượng tối đa(GB)'.")
        return
    excludeStr = exclude_str_text.value

    with output:
        print("Đã lưu các giá trị đầu vào!")
        print(f"Your Drive Link: {destDriveLink}")
        print(f"Shared Drive Link: {sourceDriveLink}")
        print(f"From Page: {fromPage}")
        print(f"To Page: {toPage}")
        print(f"Max Download Size (GB): {maxDownloadSize}")
        print(f"Exclude String: {excludeStr}")

    # Call the new copy function here after inputs are saved
    perform_copy_with_progress()

start_button.on_click(on_start_button_clicked)

display(dest_text)
display(source_text)
display(from_page_text)
display(to_page_text)
display(max_download_size_text)
display(exclude_str_text)
display(start_button, output)

Text(value='', description='Your drive', placeholder='Nhập đường link folder Google Drive của bạn')

Text(value='', description='Shared drive', placeholder='Nhập đường link folder Google Drive shared')

Text(value='0', description='Từ trang')

Text(value='0', description='Đến trang')

Text(value='700', description='Tổng dung lượng tối đa(GB)')

Text(value='', description='Bỏ file, folder có chứa nội dung')

Button(description='Bắt đầu', style=ButtonStyle())

Output()

### Chuẩn bị cho việc sao chép: Cài đặt và Xác thực Google Drive API

Để sao chép các tệp và thư mục từ Google Drive này sang Google Drive khác với các tiêu chí lọc (như trang, dung lượng tối đa, loại trừ chuỗi), chúng ta cần tương tác sâu hơn với Google Drive API. Thư viện `PyDrive` sẽ giúp đơn giản hóa việc này. Dưới đây là các bước:

1.  **Cài đặt PyDrive**: Cài đặt thư viện Python cho Google Drive API.
2.  **Xác thực**: Xác thực tài khoản Google Drive của bạn để cho phép Colab truy cập vào các tệp của bạn.
3.  **Hàm sao chép**: Viết hàm chính để xử lý logic sao chép, bao gồm hiển thị tiến trình.

In [28]:
# 1. Cài đặt PyDrive
%pip install PyDrive

In [31]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials # Re-introducing for explicit credential conversion
import os
import re
import time

def authenticate_drive():
    auth.authenticate_user()
    gauth = GoogleAuth()
    # Explicitly tell PyDrive not to look for client_secrets.json
    gauth.settings['client_config_file'] = ''

    # Get credentials from Colab's auth
    colab_creds = auth.get_user_credentials()

    # Convert to oauth2client.client.GoogleCredentials format for PyDrive
    gauth.credentials = GoogleCredentials(
        access_token=colab_creds.token,
        client_id=None, # Not typically needed for colab's interactive flow with PyDrive
        client_secret=None, # Not typically needed
        refresh_token=colab_creds.refresh_token,
        token_expiry=colab_creds.expiry,
        token_uri='https://oauth2.googleapis.com/token',
        user_agent=None,
        scopes=colab_creds.scopes
    )
    # No need for gauth.Authorize() if credentials are explicitly set
    drive = GoogleDrive(gauth)
    return drive

# Helper function to get folder ID from a Google Drive URL
def get_folder_id_from_url(url):
    match = re.search(r'drive.google.com/drive/folders/([a-zA-Z0-9_-]+)', url)
    if match:
        return match.group(1)
    match = re.search(r'file/d/([a-zA-Z0-9_-]+)', url)
    if match:
        return match.group(1)
    return url # Assume it's already an ID if no pattern matched

class StopCopyingException(Exception):
    pass

# Global variables for tracking progress
total_copied_size_bytes = 0
total_copied_files = 0

def perform_copy_with_progress():
    global destDriveLink, sourceDriveLink, fromPage, toPage, maxDownloadSize, excludeStr, output
    global drive_api, total_copied_size_bytes, total_copied_files

    with output:
        output.clear_output() # Clear previous output for a clean run
        print("--- Bắt đầu quá trình sao chép ---\n")
        print(f"Nguồn Shared Drive: {sourceDriveLink}")
        print(f"Đích Your Drive: {destDriveLink}")
        print(f"Trang bắt đầu: {fromPage}, Trang kết thúc: {toPage}")
        print(f"Dung lượng tối đa (GB): {maxDownloadSize}")
        print(f"Bỏ qua chuỗi: '{excludeStr}'")

        total_copied_size_bytes = 0 # Reset for new run
        total_copied_files = 0 # Reset for new run
        max_download_size_bytes = maxDownloadSize * 1024 * 1024 * 1024 # Convert GB to bytes

        # 1. Gắn kết Google Drive (cho đích nếu cần đường dẫn cục bộ)
        from google.colab import drive
        try:
            drive.mount('/content/drive', force_remount=True)
            print("\nGoogle Drive đã được gắn kết vào '/content/drive'.")
        except Exception as e:
            print(f"\nLỗi khi gắn kết Google Drive: {e}. Có thể không ảnh hưởng nếu chỉ dùng Drive API.\n")

        # 2. Xác thực PyDrive
        print("Đang xác thực PyDrive...")
        try:
            drive_api = authenticate_drive()
            print("Xác thực PyDrive thành công.\n")
        except Exception as e:
            print(f"\nLỗi khi xác thực PyDrive: {e}. Vui lòng kiểm tra quyền truy cập của bạn.\n")
            return

        source_folder_id = get_folder_id_from_url(sourceDriveLink)
        dest_folder_id = get_folder_id_from_url(destDriveLink)

        if not source_folder_id:
            print(f"Không thể tìm thấy ID thư mục nguồn từ liên kết: {sourceDriveLink}\n")
            return
        if not dest_folder_id:
            print(f"Không thể tìm thấy ID thư mục đích từ liên kết: {destDriveLink}\n")
            return

        print(f"Bắt đầu liệt kê và sao chép các file/thư mục từ nguồn (ID: {source_folder_id}) đến đích (ID: {dest_folder_id})...\n")

        def copy_item_recursive(source_item_id, source_item_title, dest_parent_id, current_level=0):
            # Changed from 'nonlocal' to 'global' for these variables
            global total_copied_size_bytes, total_copied_files

            indent = "  " * (current_level + 1)

            # Check for exclusion string
            if excludeStr and excludeStr.lower() in source_item_title.lower():
                print(f"{indent}Bỏ qua '{source_item_title}' (ID: {source_item_id}) vì chứa chuỗi loại trừ: '{excludeStr}'")
                return

            try:
                source_item = drive_api.CreateFile({'id': source_item_id})
                source_item.FetchMetadata(fields='mimeType,fileSize,title')

                if source_item['mimeType'] == 'application/vnd.google-apps.folder':
                    # It's a folder, create it in the destination and copy its contents
                    print(f"{indent}Đang tạo thư mục: '{source_item_title}'")
                    new_folder = drive_api.CreateFile({
                        'title': source_item_title,
                        'mimeType': 'application/vnd.google-apps.folder',
                        'parents': [{'id': dest_parent_id}]
                    })
                    new_folder.Upload() # This creates the folder on Drive
                    print(f"{indent}  Đã tạo thư mục '{source_item_title}' (ID: {new_folder['id']}). Bắt đầu sao chép nội dung.")

                    # Recursively copy contents of this folder
                    list_and_copy_filtered_contents(source_item_id, new_folder['id'], current_level + 1)

                else:
                    # It's a file, copy it directly using PyDrive's copy method
                    file_size = int(source_item['fileSize']) if 'fileSize' in source_item and source_item['fileSize'] else 0

                    if max_download_size_bytes > 0 and (total_copied_size_bytes + file_size) > max_download_size_bytes:
                        print(f"\n{indent}Dừng sao chép. Sao chép '{source_item_title}' sẽ vượt quá dung lượng tải tối đa ({maxDownloadSize} GB).")
                        raise StopCopyingException("Max download size reached.")

                    print(f"{indent}Đang sao chép file: '{source_item_title}' (Kích thước: {file_size / (1024*1024):.2f} MB)")

                    # Copy the file
                    copied_file = source_item.Copy(parameters={'parents': [{'id': dest_parent_id}]})
                    total_copied_size_bytes += file_size
                    total_copied_files += 1
                    print(f"{indent}  Đã sao chép '{source_item_title}' thành công. (ID đích: {copied_file['id']}).")
                    print(f"{indent}  Tổng files đã sao chép: {total_copied_files}. Tổng dung lượng: {total_copied_size_bytes / (1024*1024*1024):.2f} GB")
                    time.sleep(0.1) # Small delay for better progress visibility

            except StopCopyingException as e:
                raise e # Re-raise to stop parent calls
            except Exception as e:
                print(f"{indent}Lỗi khi xử lý '{source_item_title}' (ID: {source_item_id}): {e}")

        def list_and_copy_filtered_contents(current_source_folder_id, current_dest_folder_id, current_level=0):
            q_query = f"'{current_source_folder_id}' in parents and trashed=false"

            # Retrieve all files/folders in the current source folder
            all_items_in_folder = drive_api.ListFile({'q': q_query}).GetList()

            # Apply page filters (fromPage, toPage) only for the top-level call
            items_to_process = []
            if current_level == 0 and (fromPage > 0 or toPage > 0):
                start_index = fromPage - 1 if fromPage > 0 else 0
                end_index = toPage if toPage > 0 else len(all_items_in_folder)
                items_to_process = all_items_in_folder[start_index:end_index]
                print(f"  Chỉ sao chép từ mục {start_index+1} đến mục {end_index} trong thư mục hiện tại.")
            else:
                items_to_process = all_items_in_folder

            for item_idx, file_item in enumerate(items_to_process):
                try:
                    copy_item_recursive(file_item['id'], file_item['title'], current_dest_folder_id, current_level)
                except StopCopyingException as e:
                    print(e)
                    return # Stop further copying in this recursion level
                except Exception as e:
                    print(f"  Lỗi khi xử lý mục trong danh sách: {file_item.get('title', 'Unknown Title')} (ID: {file_item['id']}): {e}")


        try:
            list_and_copy_filtered_contents(source_folder_id, dest_folder_id, current_level=0)
        except StopCopyingException as e:
            print(e)
        except Exception as e:
            print(f"\nLỗi tổng quát trong quá trình sao chép: {e}\n")


        print("\n--- Quá trình sao chép hoàn tất ---")
        print(f"Tổng số file/thư mục đã sao chép: {total_copied_files}")
        print(f"Tổng dung lượng đã sao chép: {total_copied_size_bytes / (1024*1024*1024):.2f} GB")